In [ ]:
from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
SEED = CONFIG['seed']
OUTPUT.mkdir(exist_ok=True)
DECIMALS = 3
PREFIXES = {'empirical_complete':'DBLP', 'arw4':'ARWFour', 'lognormal_grw':'GRW', 'ar1_s':'AROne',
            'hurdle_ar1_s_p':'HurdleAROne', 'ar6_6':'ARSix', 'se_hurdle_s':'SEHurdle'}
WINDOWS = {'last_four':'FourYear', 'career_total':'Career'}
TOTAL_METRICS = [('QQRtwo','qq_r2',DECIMALS,1), ('MeanTotal','mean_total',DECIMALS,1),
                 ('Gini','gini_total',DECIMALS,1), ('FitKSD','ks_D',DECIMALS,1),
                 ('NTotal','n_total',0,1), ('NPositive','n_positive',0,1),
                 ('NZero','n_zero',0,1), ('ZeroPercent','zero_fraction',1,100)]
MOMENT_METRICS = [('MedianTotal','median'), ('SDTotal','sd'), ('VarianceTotal','variance')]
RANK_RMSE_METRICS = [('RankInitialRMSE','age_0_rmse11_20'), ('RankLagThreeRMSE','lag_3_rmse11_20'),
                     ('RankLagFiveRMSE','lag_5_rmse11_20')]
RANK_METRICS = [('RankInitialFinal','age_0_rho20'), ('RankLagThreeFinal','lag_3_rho20'),
                ('RankLagFiveFinal','lag_5_rho20')]
FILES = {'lognormality':INPUT/'lognormality_summary.csv',
         'moments':INPUT/'historic_distribution_summary.csv',
         'comparison':INPUT/'model_comparison.csv',
         'ranks':INPUT/'rank_persistence_slopes.csv', 'config':ROOT/'config.json'}
FILES.update({'annual':INPUT/'curve_agreement.csv', 'restart':INPUT/'reentry_agreement.csv',
              'dropout':INPUT/'dropout_agreement.csv', 'paths':INPUT/'path_agreement.csv',
              'wasserstein':INPUT/'annual_wasserstein_summary.csv',
              'wasserstein_stages':INPUT/'stage_wasserstein_summary.csv'})

In [ ]:
totals = pd.read_csv(FILES['lognormality']).set_index(['source','window'])
moments = pd.read_csv(FILES['moments']).set_index(['source','diagnostic'])
comparison = pd.read_csv(FILES['comparison']).set_index('source')
rank_start, rank_end = totals.loc[('empirical_complete','last_four'), ['start_age','end_age']]
slopes = pd.read_csv(FILES['ranks']).set_index(['source','reference']).slope
rank_references = {'age_0':'RankInitialFourYearSlope', 'lag_5':'RankLagFiveFourYearSlope'}

In [ ]:
rows = []
def add_macro(name, value, table, source, window, statistic, decimals=DECIMALS, scale=1):
    if name.endswith(('MeanTotal', 'MedianTotal')):
        decimals = 0
    elif name.endswith('SDTotal'):
        decimals = 1
    elif name.endswith('Gini'):
        decimals = 3
    grouping = ',' if statistic in ('n_total', 'n_positive', 'mean_total', 'median', 'sd') else ''
    rows.append({'macro':name, 'table':table, 'source':source, 'window':window, 'statistic':statistic,
                 'raw_value':value, 'scale':scale, 'decimals':decimals,
                 'formatted_value':f'{value*scale:{grouping}.{decimals}f}', 'alias_of':''})

empirical = totals.loc[('empirical_complete','last_four')]
for name,column in [('DBLPNComplete','n_total'), ('DBLPFourYearStart','start_age'), ('DBLPFourYearEnd','end_age')]:
    add_macro(name,empirical[column],'lognormality','empirical_complete','last_four',column,0)
add_macro('ModelSeed',SEED,'config','','','seed',0)
for source,prefix in PREFIXES.items():
    for window,window_label in WINDOWS.items():
        selected = totals.loc[(source,window)]
        for suffix,column,decimals,scale in TOTAL_METRICS:
            add_macro(prefix+window_label+suffix,selected[column],'lognormality',source,window,column,decimals,scale)
        for suffix,column in MOMENT_METRICS:
            add_macro(prefix+window_label+suffix,moments.loc[(source,window),column],'moments',source,window,column)
    for reference,suffix in rank_references.items():
        add_macro(prefix+suffix,slopes.loc[(source,reference)],'ranks',source,
                  f'annual_{int(rank_start)}_{int(rank_end)}',reference,4)
    selected = comparison.loc[source]
    if source != 'empirical_complete':
        for suffix,column in RANK_RMSE_METRICS:
            add_macro(prefix+suffix,selected[column],'comparison',source,'annual_11_20',column,4)
    for suffix,column in RANK_METRICS:
        add_macro(prefix+suffix,selected[column],'comparison',source,'annual_final',column)
    add_macro(prefix+'FinalYearZeroPercent',selected.final_zero_fraction,'comparison',source,'annual_final',
              'final_zero_fraction',1,100)
catalog = pd.DataFrame(rows)

In [ ]:
# short QQ names always refer to the primary four-year window
aliases = {prefix+'QQRtwo':prefix+'FourYearQQRtwo' for prefix in PREFIXES.values()}
aliases['GRWLognormalRtwo'] = 'GRWFourYearQQRtwo'
alias_rows = []
for name,target in aliases.items():
    row = catalog.set_index('macro').loc[target].to_dict()
    alias_rows.append({'macro':name,**row,'alias_of':target})
catalog = pd.concat([catalog,pd.DataFrame(alias_rows)],ignore_index=True)

In [ ]:
annual = pd.read_csv(FILES['annual']).set_index(['source','window','domain','scale','statistic'])
restart = pd.read_csv(FILES['restart']).set_index('source')
paths = pd.read_csv(FILES['paths']).set_index(['source','window','descriptor'])
w1 = pd.read_csv(FILES['wasserstein']).set_index('source')

stage_w1 = pd.read_csv(FILES['wasserstein_stages']).set_index(['source','stage'])
dropout_rmse = pd.read_csv(FILES['dropout']).set_index('source').weighted_rmse

In [ ]:
rows = []
def add_trajectory_macro(prefix, suffix, value, table, selection, column, decimals, scale=1):
    if suffix in ('AnnualWassersteinMean', 'AnnualWassersteinSlope'):
        decimals = 2
    if pd.notna(value):
        rows.append({'macro':prefix+suffix, 'table':table, 'selection':json.dumps(selection),
                     'statistic':column, 'raw_value':value, 'scale':scale, 'decimals':decimals, 'alias_of':'',
                     'formatted_value':f'{value*scale:.{decimals}f}'})

for source,prefix in PREFIXES.items():
    if source == 'empirical_complete':
        continue
    for suffix,column in [('AnnualWassersteinMean','mean_wasserstein1'),
                          ('AnnualWassersteinSlope','slope_wasserstein1')]:
        add_trajectory_macro(prefix,suffix,w1.loc[source,column],'wasserstein',{'source':source},column,4)
    for stage, stage_row in stage_w1.loc[source].iterrows():
        start, end = int(stage_row.start_age), int(stage_row.end_age)
        add_trajectory_macro(prefix,'AnnualWasserstein'+stage+'Mean',stage_w1.loc[(source,stage),'mean_wasserstein1'],
                  'wasserstein_stages',{'source':source,'start_age':start,'end_age':end},'mean_wasserstein1',2)
    for statistic in ['mean','median','variance']:
        key = (source,'common_generated','annual','raw',statistic)
        fields = ('source','window','domain','scale','statistic')
        add_trajectory_macro(prefix,'Annual'+statistic.title()+'NRMSEPercent',annual.loc[key,'nrmse'],
                  'annual',dict(zip(fields,key)),'nrmse',2,100)
    add_trajectory_macro(prefix,'DropoutRMSEPoints',dropout_rmse.loc[source],
              'dropout',{'source':source,'stage':'pooled'},'weighted_rmse',2,100)
    add_trajectory_macro(prefix,'RestartHazardRMSEPoints',restart.loc[source,'risk_weighted_rmse'],
              'restart',{'source':source},'risk_weighted_rmse',2,100)
    key = (source,'common_generated','longest_observed_zero_run')
    add_trajectory_macro(prefix,'LongestInactiveSpellKSD',paths.loc[key,'ks_D'],'paths',
              dict(zip(('source','window','descriptor'),key)),'ks_D',4)

trajectory = pd.DataFrame(rows)
trajectory['source'] = trajectory.selection.map(lambda selection: json.loads(selection)['source'])
catalog = pd.concat([catalog,trajectory],ignore_index=True)

In [ ]:
# only the model comparison table defines primary result tags
paper = (ROOT.parent/'manuscript/paper/report_main.tex').read_text()
tables = re.findall(r'\\begin\{table\*?\}.*?\\end\{table\*?\}', paper, flags=re.S)
comparison_table = next((table for table in tables if r'\label{tab:model-comparison}' in table), '')
primary_macros = set(re.findall(r'\\([A-Za-z]+)', comparison_table))
catalog['tag'] = catalog.macro.map(lambda name: name if name in primary_macros else 'si_'+name)
# record both the input link and its authoritative CSV; no researcher-level data
provenance = pd.DataFrame([{'table':key, 'input_path':str(path.relative_to(ROOT.parent)),
                            'source_file':str(path.resolve().relative_to(ROOT.parent)),
                            'sha256':hashlib.sha256(path.read_bytes()).hexdigest()} for key,path in FILES.items()])
manifest = catalog.merge(provenance,on='table')
manifest.to_csv(OUTPUT/'macro_manifest.csv',index=False)
provenance.to_csv(OUTPUT/'report_sources.csv',index=False)
header = [r'% generated by dblp_models/g_report/src/report.ipynb; refresh with make report',
          r'% FourYear = summed final four years; Career = whole-career supplement',
          r'% QQ and FitKSD use positive totals; raw moments and Gini include zeros',
          r'% Rank RMSE uses annual rank curves over endpoint years 11--20',
          r'% W1 means and slopes use raw annual distances',
          r'% Dropout and restart RMSE use empirical risk weights',
          r'% Percent values omit the percent sign']
definitions = []

for row in catalog.itertuples():
    value = '\\'+row.alias_of if row.alias_of else row.formatted_value
    if row.tag != row.macro:
        definitions.append(r'\expandafter\newcommand\csname '+row.tag+r'\endcsname{'+value+r'\xspace}')
        value = r'\csname '+row.tag+r'\endcsname'
    definitions.append('\\newcommand{\\'+row.macro+'}{'+value+'\\xspace}')

# atomic replacement while an editor may be compiling the manuscript
temporary = OUTPUT/'results.tex.tmp'
temporary.write_text('\n'.join(header+definitions)+'\n')
temporary.replace(OUTPUT/'results.tex')

In [ ]:
# one row per reported statistic, one column per primary model or observed reference
labels = {'empirical_complete':'dblp', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'H-AR(1)', 'se_hurdle_s':'SE-H'}
labels = {source:labels[source] for source in ['empirical_complete',*CONFIG['primary_plot_models']]}
comparison = catalog.loc[catalog.alias_of.eq(''),['macro','source','raw_value','scale']]
comparison = comparison.loc[comparison.source.isin(labels)].copy()
comparison['result'] = [macro.removeprefix(PREFIXES[source]) for macro,source in
                        zip(comparison.macro,comparison.source)]
comparison['value'] = comparison.raw_value * comparison.scale
comparison['model'] = comparison.source.map(labels)
wide = comparison.pivot(index='result',columns='model',values='value')
wide = wide.reindex(index=comparison.result.drop_duplicates(),columns=labels.values())
wide.to_csv(OUTPUT/'primary_model_comparison.csv')